# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List available record sets and their fields
print("Available Record Sets and their fields (identified by @id):\n")
for rs in metadata.record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, there is typically only one record set for tabular data (e.g., protein analysis table).
# We'll extract the relevant record set and display its columns and a few sample records.

# ---
# Find the appropriate record set @id (usually the main table)
record_sets = [rs['@id'] for rs in metadata.record_sets]
# For this dataset, the main @id is likely: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034:protein' (assuming a 'protein' table)
# You may inspect the cells above to get the exact value if available. For this implementation, we'll infer it as the first one.
print('Record Sets:', record_sets)

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}")

# Display columns of the first record set (main analysis table)
main_record_set_id = record_sets[0]
print('Record set columns:', dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field for filtering/normalization
# Let's assume there is a field representing 'Molecular Weight' with @id 'protein_mw'.
# (Check the output from section 2 for exact field names or ids, here we use a typical one for proteomics.)
df = dataframes[main_record_set_id]
candidate_numeric_fields = [col for col in df.columns if ('weight' in col.lower()) or ('abundance' in col.lower()) or ('count' in col.lower())]
print('Candidate numeric fields:', candidate_numeric_fields)
# We'll choose the first candidate for demonstration.
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # Fallback to first column
    numeric_field = df.columns[0]
print('Selected numeric_field:', numeric_field)

# Try to coerce to numeric in case numbers are strings
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filtering records with value > threshold (e.g., MW > 10 kDa)
threshold = 10000 if 'weight' in numeric_field.lower() else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by protein family/annotation if such a field exists
group_fields = [col for col in df.columns if any(x in col.lower() for x in ['family', 'type', 'group', 'accession', 'description'])]
group_field = group_fields[0] if group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Scatterplot if at least two numeric fields exist
if len(candidate_numeric_fields) >= 2:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[candidate_numeric_fields[0]], y=df[candidate_numeric_fields[1]])
    plt.xlabel(candidate_numeric_fields[0])
    plt.ylabel(candidate_numeric_fields[1])
    plt.title(f"Scatterplot: {candidate_numeric_fields[0]} vs {candidate_numeric_fields[1]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, you have seen how to load and explore a FAIR-compliant proteomics dataset using the `mlcroissant` library.
* We listed all record sets and fields, extracted the main protein analysis table, examined numeric fields (like molecular weight or abundance), filtered and normalized the results, grouped by biological annotation, and visualized distributions.
* These templates and dynamic lookups respect all Croissant `@id` conventions and allow you to extend easily to additional datasets of similar structure.